# 🏛️ 02 - SÉNATEUR V2 : POISSON SUPREMACY (L'Expert)

Ce notebook implémente le deuxième membre du "Grand Conseil" : **Divination V2**.

### 🎯 Stratégie :
- Remplacer le **GradientBoosting** (trop corrélé au LightGBM) par un modèle radicalement différent : **Poisson Regression**.
- Objectif : Apporter un point de vue "statistique" et conservateur au vote.
- Ce modèle excelle en **Précision** (peu de faux positifs).

### ⚙️ Composition :
- **XGBoost 1 & 2** (Poids 0.35 chacun)
- **LightGBM** (Poids 0.40)
- **Poisson Supremacy** (Poids 0.35) 🆕
- **LogisticRegression** (Poids 0.12)

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

In [ ]:
# --- Class Poisson Supremacy ---
class PoissonSupremacyClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper pour transformer une régression de Poisson en Classifieur"""
    
    def __init__(self, learning_rate=0.05, max_iter=500, max_depth=8, l2_regularization=0.1, random_state=42):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.max_depth = max_depth
        self.l2_regularization = l2_regularization
        self.random_state = random_state
    
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.model_ = HistGradientBoostingRegressor(
            loss='poisson',
            learning_rate=self.learning_rate,
            max_iter=self.max_iter,
            max_depth=self.max_depth,
            l2_regularization=self.l2_regularization,
            random_state=self.random_state
        )
        self.model_.fit(X, y)
        return self
    
    def predict_proba(self, X):
        pred = self.model_.predict(X)
        # Clip pour rester entre [0,1]
        proba_1 = np.clip(pred, 0, 1)
        return np.column_stack([1 - proba_1, proba_1])
    
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

## 1. Chargement & Pipeline

In [ ]:
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')

def feature_engineering(df):
    df_eng = df.copy()
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

X = feature_engineering(train_df.drop('converted', axis=1))
y = train_df['converted']
X_test = feature_engineering(test_df)

# Preprocessor Standard
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['age', 'total_pages_visited', 'interaction_age_pages', 'pages_per_age']),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['country', 'source'])
    ],
    remainder='passthrough'
)

# Voting avec Poisson
voting_clf = VotingClassifier(
    estimators=[
        ('xgb1', XGBClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.9, random_state=42)),
        ('xgb2', XGBClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.9, random_state=2025)),
        ('lgbm', LGBMClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.9, verbose=-1, random_state=42)),
        ('poisson', PoissonSupremacyClassifier()),
        ('logreg', LogisticRegression(class_weight={0: 1, 1: 80}))
    ],
    voting='soft',
    weights=[0.35, 0.35, 0.40, 0.35, 0.12], # Poids Optimaux V2
    n_jobs=-1
)

pipeline = Pipeline([('pre', preprocessor), ('clf', voting_clf)])

## 2. Validation Rapide (5-Fold)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))

print("🚀 Cross-Validation...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    pipeline.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof_preds[val_idx] = pipeline.predict_proba(X.iloc[val_idx])[:, 1]
    print(f"   Fold {fold}/5 terminé.")

# Optimisation Seuil
best_thr, best_f1 = 0.5, 0
for thr in np.linspace(0.3, 0.6, 100):
    score = f1_score(y, (oof_preds >= thr).astype(int))
    if score > best_f1: best_f1, best_thr = score, thr

print(f"✅ V2 F1-Score : {best_f1:.5f} (Seuil : {best_thr:.3f})")

In [ ]:
# Entraînement Final
pipeline.fit(X, y)
test_preds = (pipeline.predict_proba(X_test)[:, 1] >= best_thr).astype(int)

pd.DataFrame({'converted': test_preds}).to_csv('divination_v2_predictions.csv', index=False)
print(f"💾 Prédictions V2 sauvegardées ({test_preds.sum()} conversions)")